# Validating CoLoRe simulation for CMB lensing

- The CoLoRe simulation boxes were first generated by Laura Casas
- The notebook just uses the CMB lensing convergence $\kappa$ values of the boxes and validates it through the auto-spectrum.
- The kappa values can be found in `/data/.../`, along with $C_{\ell}$ (and its associated errors), and a logarithmic rebinning run with `scripts/rebin_cls_kk.py`, which is the one used in this notebook.
- The notebook also compares the two CoLoRe boxes in the folder.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import camb
from camb import sources
import scipy
from scipy.integrate import quad
from scipy.constants import c
import sys
sys.path.append('../')

from py.rebinning import log_bin_cls  
import json

plt.rcParams.update({
    "text.usetex": True,
    "font.family": 'serif',
})

# Theory CMB lensing auto-spectrum $C_{\ell}^{\kappa \kappa}$

## setting up CAMB

In [2]:
# set the k and redshift range for the boltzmann solver
kmin, kmax = 1E-3, 100
zmin, zmax = 0, 1100
nz = 200
z_pk = zmin + np.exp(np.log(zmax - zmin + 1)*np.linspace(0, 1, nz)) - 1

# range of multipoles of the power spectrum to be studied
lmin = 2
lmax = 2500
ells = np.arange(lmin, lmax+1)

In [3]:
def load_cosmology(filename):
    with open(filename) as f:
        return json.load(f)

In [4]:
def get_camb_results(
    cosmo_dict,
    z_pk,
    kmax=100.0,
    lmax=2500,
    nonlinear=False,
    window=False,
):
    """
    Read a cosmological parameter dictionary, extract cosmological parameters,
    and build:

    1. CAMBparams instance
    2. CAMB results object

    Parameters
    ----------
    cosmo_dict : dict
        Dictionary containing cosmological parameters

    z_pk : list or None
        Redshifts for P(k) interpolator.
        If None, defaults to [0.0]

    kmax : float
        Maximum k [1/Mpc]

    lmax : integer
        Maximum multipole

    nonlinear : bool
        Whether to use nonlinear P(k)
    """

    # --------------------------------------------------
    # Convert to CAMB inputs
    # --------------------------------------------------

    H0 = 100.0 * cosmo_dict["h"]
    ombh2 = cosmo_dict["omega_B"] * cosmo_dict["h"]**2
    omch2 = (cosmo_dict["omega_M"] - cosmo_dict["omega_B"]) * cosmo_dict["h"]**2

    # --------------------------------------------------
    # Initial CAMB params
    # --------------------------------------------------

    pars = camb.CAMBparams()

    pars.set_cosmology(
        H0=H0,
        ombh2=ombh2,
        omch2=omch2,
    )

    pars.set_dark_energy(w=cosmo_dict["w"])
    pars.InitPower.set_params(
        ns=cosmo_dict["ns"],
        As=cosmo_dict["As"]
    )

    # --------------------------------------------------
    # Matter power settings
    # --------------------------------------------------

    pars.set_matter_power(
        redshifts=z_pk,
        kmax=kmax
    )

    if nonlinear:
        pars.NonLinear = camb.model.NonLinear_both
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=2)
    else:
        pars.set_for_lmax(lmax=lmax, lens_potential_accuracy=0)

    if window:
        z_cut = cosmo_dict["z_max"]
        z = np.linspace(0.01, z_cut, 200)
        W = np.zeros_like(z)
        W[W < 0] = 0
        win = sources.SplinedSourceWindow(
            z=z,
            W=W,
            source_type='lensing'
        )
        pars.SourceWindows = [win]
        pars.Want_CMB = False
        pars.Want_cl_2D_array = True


    # Final results
    results = camb.get_results(pars)

    return results

In [5]:
# get linear and non-linear matter power spectrum
cosmo_dict = load_cosmology("../data/second_test_box/cosmology.json")
z_max, z_out = cosmo_dict["z_max"], cosmo_dict["z_out"]
results_lin = get_camb_results(
    cosmo_dict=cosmo_dict,
    z_pk=z_pk,
    kmax=kmax,
    lmax=lmax,
    nonlinear=False)
results_nlin = get_camb_results(
    cosmo_dict=cosmo_dict,
    z_pk=z_pk,
    kmax=kmax,
    lmax=lmax,
    nonlinear=True)

Note: redshifts have been re-sorted (earliest first)
Note: redshifts have been re-sorted (earliest first)


In [ ]:
results_window = get_camb_results(
    cosmo_dict=cosmo_dict,
    z_pk=z_pk,
    kmax=kmax,
    lmax=lmax,
    nonlinear=False,
    window=True)

Note: redshifts have been re-sorted (earliest first)


## manual Limber integral

In [ ]:
# comoving distance of the last scattering surface
z_star = z_out
chi_star = results_lin.comoving_radial_distance(z_star)
chi_max = results_lin.comoving_radial_distance(z_max)  # comoving distance of z_max
H0 = cosmo_dict["h"] * 100
om = cosmo_dict["omega_M"]

In [ ]:
pk_interp_lin = results_lin.get_matter_power_interpolator(nonlinear=False, 
                                                      hubble_units=False, 
                                                      k_hunit=False,  
                                                      var1='delta_tot', 
                                                      var2='delta_tot')
pk_interp_nlin = results_nlin.get_matter_power_interpolator(nonlinear=True, 
                                                         hubble_units=False, 
                                                         k_hunit=False,  
                                                         var1='delta_tot', 
                                                         var2='delta_tot')

In [ ]:
def w_kappa(chi, z_chi, chi_source):
    '''
    CMB lensing convergence kernel
    '''
    
    # scale factor at redshift of lensing structure
    a = 1.0 / (1.0 + z_chi)
    c_kms = c*1E-3
    
    # prefactor of the kernel
    w_prefactor = (1.5) * (H0/c_kms)**2 * om * (chi_source - chi)/chi_source
    w_geom = (chi/a)
    
    return w_prefactor * w_geom

In [ ]:
def limber(chi, ell, chi_source, pk_interp, results):
    '''
    CMB lensing auto-spectrum using Limber approximation
    '''

    # converting chi to redshift
    z_chi = results.redshift_at_comoving_radial_distance(chi)
    k = (ell + 0.5) / chi  
    kp = 0.31  # k pressure parameter scale used for smoothing the power spectrum according to the colore simulation resolution


    # kernel and pk interpolator, filtering out of extrapolation k and small k
    if 1e-4 < k < kmax:
        win = w_kappa(chi, z_chi, chi_source)
        pk = pk_interp.P(z_chi, k, grid=False)
        smooth = np.exp(-(k/kp)**2)  # smoothing factor to avoid oscillations at high k 
        return (win**2 / chi**2) * pk * smooth
        
    else:
        return 0

In [ ]:
#store the cls from linear and nonlinear limber approximation integration
# redshift integration up to zmax
cl_kk_lin_zmax_zs = np.zeros(ells.shape)
cl_kk_nlin_zmax_zs = np.zeros(ells.shape)
cl_kk_lin_zmax_zmax = np.zeros(ells.shape)
cl_kk_nlin_zmax_zmax = np.zeros(ells.shape)

# manually integrate the limber approximation
for i, ell in enumerate(ells):
    
    # integral up to redshift z_max, source plane at z_star
    cl_kk_lin_zmax_zs[i] = quad(limber, 0.001, chi_max, args=(ell, chi_star, pk_interp_lin, results_lin), epsabs=1E-10)[0]
    cl_kk_nlin_zmax_zs[i] = quad(limber, 0.001, chi_max, args=(ell, chi_star, pk_interp_nlin, results_nlin), epsabs=1E-10)[0]

    # integral up to redshift z_max, source plane at z_max
    cl_kk_lin_zmax_zmax[i] = quad(limber, 0.001, chi_max, args=(ell, chi_max, pk_interp_lin, results_lin), epsabs=1E-10)[0]
    cl_kk_nlin_zmax_zmax[i] = quad(limber, 0.001, chi_max, args=(ell, chi_max, pk_interp_nlin, results_nlin), epsabs=1E-10)[0]

## full sky (non-Limber) integral

In [ ]:
# compute the lens potential cls from camb using full curved sky without Limber approx
cl_kk_camb_lin = results_lin.get_lens_potential_cls(lmax=lmax, raw_cl=True)[2:, 0]
cl_kk_camb_lin *= ((ells * (ells + 1.0)) / 2) ** 2  # converting cls_potential to cls_kappa
cl_kk_camb_nlin = results_nlin .get_lens_potential_cls(lmax=lmax, raw_cl=True)[2:, 0]
cl_kk_camb_nlin *= ((ells * (ells + 1.0)) / 2) ** 2 # converting cls_potential to cls_kappa

# CoLoRe simulation auto-spectrum $C_{\ell}^{\kappa \kappa}$

In [ ]:
first_test_box_cls_log = np.load(f'../data/first_test_box/cl_kk_log.npz')
second_test_box_cls_log = np.load(f'../data/second_test_box/cl_kk_log.npz')
ells_colore, cl_kk_colore_zmax, err_tt_zmax = first_test_box_cls_log['ells'], first_test_box_cls_log['cls'], first_test_box_cls_log['err']
ells_colore, cl_kk_colore_zs, err_tt_zs = second_test_box_cls_log['ells'], second_test_box_cls_log['cls'], second_test_box_cls_log['err']
nbins = ells_colore.size

In [ ]:
ells_colore = ells_colore[~np.isnan(ells_colore)]
cl_kk_colore_zmax = cl_kk_colore_zmax[~np.isnan(cl_kk_colore_zmax)]
cl_kk_colore_zs = cl_kk_colore_zs[~np.isnan(cl_kk_colore_zs)]
err_tt_zmax = err_tt_zmax[~np.isnan(err_tt_zmax)]
err_tt_zs = err_tt_zs[~np.isnan(err_tt_zs)]

# plotting

## latest CoLoRe box analysis and $\chi^2$ test

In [ ]:
ells_rebin, cl_kk_lin_zmax_zs_rebin = log_bin_cls(ells, cl_kk_lin_zmax_zs, nbins=nbins, ell_min=2, ell_max=1535)[:2]
ells_rebin, cl_kk_lin_zmax_zmax_rebin = log_bin_cls(ells, cl_kk_lin_zmax_zmax, nbins=nbins, ell_min=2, ell_max=1535)[:2]

ells_rebin = ells_rebin[~np.isnan(cl_kk_lin_zmax_zs_rebin)]
cl_kk_lin_zmax_zs_rebin = cl_kk_lin_zmax_zs_rebin[~np.isnan(cl_kk_lin_zmax_zs_rebin)]
cl_kk_lin_zmax_zmax_rebin = cl_kk_lin_zmax_zmax_rebin[~np.isnan(cl_kk_lin_zmax_zmax_rebin)]

chi2 = np.sum((cl_kk_colore_zs - cl_kk_lin_zmax_zs_rebin)**2 / err_tt_zs**2)
chi2_nu = chi2 / (len(ells_rebin))

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), height_ratios=[3, 1], dpi=120, sharex=True)

# plot the rebinned CoLoRe Cls with error bars
ax1.errorbar(ells_colore, cl_kk_colore_zs, yerr=err_tt_zs, elinewidth=0.7, capsize=1.0, fmt='C0.', markersize=5.0, label=rf'CoLoRe_box2 $(z_{{\rm max}} = {z_max}, \ z_s={z_out})$')
ax1.scatter(0, 0 , s=0, label=rf'$\chi^2_\nu = {chi2_nu:.2f}$')

# manual limber integration full redshift
ax1.plot(ells, cl_kk_lin_zmax_zs, c='C0', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star)}$)')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim(lmin, lmax)
ax1.set_ylabel(r'$C_{L}^{\kappa \kappa}$')
ax1.legend()

# plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
ax2.errorbar(ells_colore, (cl_kk_colore_zs-cl_kk_lin_zmax_zs_rebin)/cl_kk_lin_zmax_zs_rebin, 
             yerr=err_tt_zs/cl_kk_lin_zmax_zs_rebin, capsize=1.0, elinewidth=0.7, fmt='C0.', markersize=5.0)
ax2.axhline(0, c='k', ls='--', lw=0.7)
ax2.set_xscale('log')
ax2.set_xlabel(r'$L$')
ax2.set_ylabel('relative error')

fig.suptitle('CMB lensing autospectrum: Theory vs. CoLoRe boxes', y=0.93)

fig.savefig(f'plots/cls_kk_last_box.png', bbox_inches='tight')
fig.savefig(f'plots/cls_kk_last_box.pdf', bbox_inches='tight')
plt.show()

In [ ]:
chi2_per_bin = ((cl_kk_colore_zs - cl_kk_lin_zmax_zs_rebin) / err_tt_zs)**2
cumulative_chi2 = np.nancumsum(chi2_per_bin)
cumulative_dof = np.arange(1, len(cumulative_chi2) + 1)
p_values = 1 - scipy.stats.chi2.cdf(cumulative_chi2, df=cumulative_dof)

# Plotting the cumulative chi-square
fig, ax1 = plt.subplots(figsize=(6, 4), dpi=120)
ax2 = ax1.twinx()  # Create a second y-axis for p-values

# Plot your cumulative chi-square
ax1.plot(ells_rebin, cumulative_chi2, marker='.', linestyle='-', color='C0', label=r'Cumulative $\chi^2$')
ax2.plot(ells_rebin, p_values, marker='.', linestyle='-', color='C1', label=r'Cumulative $p$-value', alpha=0.3)

# Plot the expected theoretical baseline (Expected chi^2 = dof)
ax1.plot(ells_rebin, cumulative_dof, linestyle='--', color='black', label=r'Expected $\chi^2$ (Perfect Fit)')

# Formatting the plot
ax1.set_xscale('log')
ax1.set_yscale('log') # A log scale on the y-axis is highly recommended since the max value is ~8000
ax1.set_xlabel('$L$')
ax1.set_ylabel(r'Cumulative $\chi^2$')
ax2.set_ylabel(r'Cumulative $p$-value')
ax1.set_title(r'Cumulative $\chi^2$ vs. Multipole $L$')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.show()

## compare Limber vs non-Limber with the second CoLoRe box

In [ ]:
plt.figure(figsize=(6, 4), dpi=120)

# plot the rebinned CoLoRe Cls with error bars
plt.errorbar(ells_colore, cl_kk_colore_zs, yerr=err_tt_zs, elinewidth=0.7, capsize=1.0, fmt='C0.', markersize=4.0, label=rf'CoLoRe_box2 $(z_{{\rm max}} = {z_max}, \ z_s={z_out})$')


# manual limber integration full redshift
plt.loglog(ells, cl_kk_lin_zmax_zs, c='C0', ls='-', label=rf'Limber_lin ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star)}$)')
plt.loglog(ells, cl_kk_nlin_zmax_zs, c='C0', ls='--', label=rf'Limber_nonlin ($z_{{\rm cut}} = {z_max}, \ z_s={int(z_star)}$)')


# full sky integration full redshift
plt.loglog(ells, cl_kk_camb_lin, c='C1', ls='-', label=rf'nonLimber_lin ($z_{{\rm cut}} = z_s={int(z_star)}$)')
plt.loglog(ells, cl_kk_camb_nlin, c='C1', ls='--', label=rf'nonLimber_nonlin ($z_{{\rm cut}} = z_s={int(z_star)}$)')

plt.xlim(lmin, lmax)
plt.xlabel(r'$L$')
plt.ylabel(r'$C_{L}^{\kappa \kappa}$')


plt.title('CMB lensing autospectrum: limber/non-Limber Theory vs. CoLoRe')
plt.legend()

plt.savefig(f'plots/cls_kk_limber_comparison.png', bbox_inches='tight')
plt.savefig(f'plots/cls_kk_limber_comparison.pdf', bbox_inches='tight')
plt.show()

## comparing CoLoRe boxes
source plane changes between them

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), height_ratios=[3, 1], dpi=120, sharex=True)

# plot the rebinned CoLoRe Cls with error bars
ax1.errorbar(ells_colore, cl_kk_colore_zmax, yerr=err_tt_zmax, elinewidth=0.7, capsize=1.0, fmt='C1.', markersize=5.0, label=rf'CoLoRe_box1 ($z_{{\rm max}} = z_s = {z_max}$)')
ax1.errorbar(ells_colore, cl_kk_colore_zs, yerr=err_tt_zs, elinewidth=0.7, capsize=1.0, fmt='C0.', markersize=5.0, label=rf'CoLoRe_box2 ($z_{{\rm max}} = {z_max}, \ z_s={z_out}$)')

# manual limber integration full redshift
ax1.plot(ells, cl_kk_lin_zmax_zs, c='C0', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = {z_max}, \ z_s={z_star}$)')
ax1.plot(ells, cl_kk_lin_zmax_zmax, c='C1', ls='-', label=rf'Limber_linear ($z_{{\rm cut}} = z_s = {z_max}$)')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlim(2, 2500)
ax1.set_ylabel(r'$C_{L}^{\kappa \kappa}$')
ax1.legend()

# plot the relative error between the rebinned linear limber and the rebinned CoLoRe cls
ax2.errorbar(ells_colore, (cl_kk_colore_zs-cl_kk_lin_zmax_zs_rebin)/cl_kk_lin_zmax_zs_rebin, 
             yerr=err_tt_zs/cl_kk_lin_zmax_zs_rebin, capsize=1.0, elinewidth=0.7, fmt='C0.', markersize=5.0)
ax2.errorbar(ells_colore, (cl_kk_colore_zmax-cl_kk_lin_zmax_zmax_rebin)/cl_kk_lin_zmax_zmax_rebin, 
             yerr=err_tt_zmax/cl_kk_lin_zmax_zmax_rebin, capsize=1.0, elinewidth=0.7, fmt='C1.', markersize=5.0)
ax2.axhline(0, c='k', ls='--', lw=0.7)
ax2.set_xscale('log')
ax2.set_xlabel(r'$L$')
ax2.set_ylabel('relative error')

fig.suptitle('CMB lensing autospectrum: Theory vs. CoLoRe boxes', y=0.93)

fig.savefig(f'plots/cls_kk_boxes_comparison_kp.png', bbox_inches='tight')
fig.savefig(f'plots/cls_kk_boxes_comparison_kp.pdf', bbox_inches='tight')
plt.show()

## comparing Limber approximation: linear vs. nonlinear

In [ ]:
plt.figure(figsize=(6, 4), dpi=120)

# compare linear and nonlinear
plt.semilogx(ells, cl_kk_camb_lin - cl_kk_lin_zs_zs, label='linear')
plt.semilogx(ells, cl_kk_camb_nlin - cl_kk_nlin_zs_zs, label='nonlinear')
plt.axhline(0, c='C3', ls='--')

plt.xlabel(r'$L$')
plt.ylabel(r'$C_{L}^{\kappa \kappa, \ \rm non-Limber} - C_{L}^{\kappa \kappa, \ \rm Limber}$')

plt.legend()

plt.savefig(f'plots/cls_kk_full-vs-limber.png', bbox_inches='tight')
plt.savefig(f'plots/cls_kk_full-vs-limber.pdf', bbox_inches='tight')
plt.show()